<a href="https://colab.research.google.com/github/MuhammadRhakan/final_project/blob/main/3_Refined_Methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import nltk
import kagglehub
import pickle

import pandas as pd
import numpy as np

from kagglehub.datasets import KaggleDatasetAdapter
from sklearn.preprocessing import normalize, OneHotEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk import word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import *
from scipy.sparse import csr_matrix, vstack

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [2]:
course = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "hossaingh/udemy-courses", "Course_info.csv")

<ipython-input-2-193580186>:1: DeprecationWarning: load_dataset is deprecated and will be removed in a future version.
  course = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS, "hossaingh/udemy-courses", "Course_info.csv")


In [3]:
def missing_values(course):
  null_values = course[(course['num_lectures'] == 0) |
                (course['content_length_min'] == 0) |
                ((course['avg_rating'] == 0) & (course['num_reviews'] > 0))].index
  clean_data = course.drop(index=null_values).dropna()

  return clean_data

In [4]:
course = course[course['language'].isin({'English'})]
course = missing_values(course)

In [5]:
#separate numeric and non-numeric attributes
def attributes(data, shift='avg_rating'):
  categorical = []
  numerical = []

  for i, cat in enumerate(data.select_dtypes(include = ['object', 'bool']).columns.values):
    categorical.append(cat)
  categorical.append(shift)

  for i, num in enumerate(data.select_dtypes(include = 'number').drop(columns='id').columns.values):
    if num != shift:
      numerical.append(num)

  return categorical, numerical


#prepare clean data
def data_cleaning(data, features, par=0.9):
  outliers_indices = set()

  for col in features:
    exclude = data[col].quantile(par)
    outliers = data[data[col] > exclude]
    outliers_indices.update(outliers.index)

  trim = data.drop(index=outliers_indices)
  transformed = trim.copy()

  transformed[['num_subscribers', 'num_reviews', 'num_comments']] = np.log1p(transformed[['num_subscribers', 'num_reviews', 'num_comments']])
  transformed[['price', 'num_lectures', 'content_length_min']] = np.sqrt(transformed[['price', 'num_lectures', 'content_length_min']])

  return trim, transformed


#feature engineering for high-cardinality categorical data
def calc_smoothed_instructor_rating(data, feature, rating='avg_rating', subscriber='num_subscribers', weight=50):
  data['engagement'] = data[rating] * data[subscriber]

  instructor_stats = data.groupby(feature).agg(
      total_rating=('engagement', 'sum'),
      total_subs=(subscriber, 'sum'))

  instructor_stats['weighted_avg'] = instructor_stats.apply(lambda row: row['total_rating'] / row['total_subs'] if row['total_subs'] > 0 else 0, axis=1)
  global_avg = data['engagement'].sum() / data[subscriber].sum()
  instructor_stats['smoothed'] = (
      (instructor_stats['total_subs'] * instructor_stats['weighted_avg'] + weight * global_avg) /
      (instructor_stats['total_subs'] + weight))

  instructor_score = data[feature].map(instructor_stats['smoothed']).astype('int64')
  instructor_score.name = 'instructor_score'

  return instructor_score


#label ordinal features to categories (nominal)
def ordinal_features_labeling(data):
  series =  data.apply(lambda x: 'Low' if x < 3 else ('Medium' if x == 3 else 'High'))

  return pd.DataFrame(series)


#combine transformed features with nominal features
def combined_features(*all_data):
  return pd.concat(all_data, axis=1)

In [6]:
categorical, numerical = attributes(course)
course_clean, course_clean_scaled = data_cleaning(course, features=numerical)
instructor_score = calc_smoothed_instructor_rating(course_clean_scaled, feature='instructor_name')

categorical_ordinal = combined_features(
    course_clean[['is_paid', 'category', 'subcategory']],
    ordinal_features_labeling(course_clean['avg_rating']),
    ordinal_features_labeling(instructor_score)
    )

In [7]:
def semantic_preprocessing_batch(batch, features, stop_words, lemmatizer):
    batch = batch[features].copy()
    for col in features:
        batch[col] = batch[col].str.lower()

    for col in features:
        batch[col] = batch[col].apply(lambda text: ' '.join([lemmatizer.lemmatize(word) for word in word_tokenize(text) if word not in stop_words]))

    return batch.apply(lambda row: ' '.join(row), axis=1)


def semantic_preprocessing(data, features, batch_size=5000):
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    vectorizer = TfidfVectorizer(max_features=1000, min_df=10, max_df=0.9, ngram_range=(1, 2), dtype=np.float32)

    combined_text = []
    for i in range(0, len(data), batch_size):
      batch = data.iloc[i:i+batch_size]
      processed = semantic_preprocessing_batch(batch, features, stop_words, lemmatizer)
      combined_text.extend(processed)

    tfidf_matrix = vectorizer.fit_transform(combined_text)
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out())

    similarity_batches = []
    for i in range(0, tfidf_matrix.shape[0], batch_size):
        batch = tfidf_matrix[i:i+batch_size]
        similarity_batch = cosine_similarity(batch, tfidf_matrix)
        similarity_batches.append(similarity_batch.astype(np.float16))

    return tfidf_matrix, np.vstack(similarity_batches)

In [8]:
semantic_result, semantic_similarities = semantic_preprocessing(course_clean_scaled, ['title', 'headline'])

In [9]:
print(f'TF-IDF Matrix:  {semantic_result.shape[0]} rows x {semantic_result.shape[1]} words')
print(f'TF-IDF Similarity Matrix:   {semantic_similarities.shape[0]} items x {semantic_similarities.shape[1]} items')

TF-IDF Matrix:  85480 rows x 1000 words
TF-IDF Similarity Matrix:   85480 items x 85480 items


In [10]:
def numerical_preprocessing(data, features, batch_size=5000, threshold=0.7, max_print=None):
  normalized_matrix = normalize(data[features])
  normalized_df = pd.DataFrame(data=normalized_matrix, columns=data[features].columns)

  similarity_batches = []

  for i in range(0, len(data), batch_size):
    batch = normalized_matrix[i:i+batch_size]
    similarity_batch = cosine_similarity(batch, normalized_matrix)
    similarity_batches.append(similarity_batch.astype(np.float16))
    similarity_matrix = np.vstack(similarity_batches)

    '''
    low_sim_indices = np.where((similarity_matrix < threshold) & (similarity_matrix < 0.999))

    results = []
    for idx, (i, j) in enumerate(zip(*low_sim_indices)):
        results.append((i, j, similarity_matrix[i, j]))
        if max_print is not None and idx + 1 >= max_print:
            break

    for i, j, sim in results:
        print(f"Item {i} vs Item {j} = Similarity: {sim:.4f}")
    '''

  return normalized_matrix, np.vstack(similarity_batches), normalized_df

In [ ]:
numeric_result, numeric_similarities, normalized_data = numerical_preprocessing(course_clean_scaled, numerical)

In [ ]:
print(f'Numeric Matrix:  {numeric_result.shape[0]} rows x {numeric_result.shape[1]} features')
print(f'Numeric Similarity Matrix:   {numeric_similarities.shape[0]} items x {numeric_similarities.shape[1]} items')

In [ ]:
def nominal_preprocessing(data, features):
  nominal = data[features].copy()
  multiattribute = pd.get_dummies(nominal, prefix=features)
  multiattribute_matrix = multiattribute.values.astype(np.float16)

  ni = multiattribute_matrix.sum(axis=1)
  nij = multiattribute_matrix @ multiattribute_matrix.T

  ni_matrix = ni[:, None]
  nj_matrix = ni[None, :]
  denom = ni_matrix + nj_matrix
  denom[denom == 0] = 1e-12  # Small epsilon to avoid division by zero

  DS = (2 * nij) / denom
  DS[np.isnan(DS)] = 0.0

  nominal_similarities = DS
  nominal_result = multiattribute_matrix

  return nominal_result, nominal_similarities

In [ ]:
def nominal_preprocessing_batchwise(data, features, batch_size=500):
  multiattribute = pd.get_dummies(data[features], prefix=features, dtype=np.float32)
  multiattribute_matrix = csr_matrix(multiattribute.values)

  ni = np.asarray(multiattribute_matrix.sum(axis=1)).flatten()
  nj = ni

  n_rows = multiattribute_matrix.shape[0]
  DS_blocks = []

  for start in range(0, n_rows, batch_size):
    end = min(start + batch_size, n_rows)
    matrix_batch = multiattribute_matrix[start:end]

    nij = matrix_batch @ multiattribute_matrix.T
    ni_batch = ni[start:end][:, None]

    denom = ni_batch + nj
    denom[denom == 0] = 1e-12

    DS_batch = (2 * nij).multiply(1 / denom)
    DS_blocks.append(DS_batch)

  DS = vstack(DS_blocks)
  return multiattribute_matrix, DS

In [ ]:
nominal_result, nominal_similarities = nominal_preprocessing_batchwise(course_clean, features=[categorical_ordinal])

In [ ]:
print(f'Numeric Matrix:  {nominal_result.shape[0]} rows x {nominal_result.shape[1]} features')
print(f'Numeric Similarity Matrix:   {nominal_similarities.shape[0]} items x {nominal_similarities.shape[1]} items')

In [9]:
variable_storage = {
    'semantic_result': semantic_result,
    'semantic_similarities': semantic_similarities,
    # 'numeric_result': numeric_result,
    # 'numeric_similarities': numeric_similarities,
    # 'nominal_result': nominal_result,
    # 'nominal_similarities': nominal_similarities
}

for name, variable in variable_storage.items():
    filename = f'{name}.pkl'
    with open(filename, 'wb') as f:
      pickle.dump(variable, f)

print("Done: Each variable saved as a separate .pkl file.")

Done: Each variable saved as a separate .pkl file.
